# 06. Multi-Layer Perceptron (MLP)

As defined in Section B.4 of the proposal, we experiment with a small MLP Neural Network trained with the Adam optimizer.
We check whether a neural network can match the tree-based models on this tabular regression task.


# 06 — PyTorch MLP with Adam

**Dataset:** Inside Airbnb — New York City (open data, 2025–2026)  
**Goal:** Train a multi-layer perceptron on the leakage-free feature matrix and generate 5-fold OOF predictions for blending.

Techniques:
- **Multi-layer Perceptron** (3 hidden layers: 256→128→64) built in PyTorch.
- **Adam optimizer** with learning rate scheduling.
- **StandardScaler** for numeric normalization.
- **K-Fold cross-validation** (`KFold(5, shuffle=True, random_state=42)`) — same split as all other models for valid OOF blending.
- Predictions clipped to `[0, 90]` (Q1 2026 has 90 calendar days).

Outputs:
- `outputs/oof_MLP.npy`
- `outputs/test_local_pred_MLP.npy`

## 1. Setup

In [12]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'  # also silences warnings from n_jobs=-1 joblib subprocesses

import warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
N_SPLITS = 5
torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

OUT_DIR = Path('../outputs'); OUT_DIR.mkdir(exist_ok=True, parents=True)
if torch.cuda.is_available():
    device = torch.device('cuda') # Nvdia GPU (for Muhammed Ali Bakır's Computer)
elif torch.backends.mps.is_available():
    device = torch.device('mps')  # Apple GPU (for MacBook Air M4 Mahmut Sami Başkal's Computer)
else:
    device = torch.device('cpu')  # CPU (Any other computer without GPU support)
print(f'PyTorch {torch.__version__} on {device}')

PyTorch 2.12.0 on mps


## 2. Load Features

In [13]:
# Section IV.C: Load Development set and Local Test set separately
train_df = pd.read_parquet(OUT_DIR / 'train_local.parquet')
test_local_df = pd.read_parquet(OUT_DIR / 'test_local.parquet')
print('Loaded parquet files from', OUT_DIR.resolve())

TARGET = 'blocked_days_Q1_2026'
ID_COL = 'id'
DROP_COLS = [ID_COL, TARGET]

y = train_df[TARGET].astype(float).values
X = train_df.drop(columns=DROP_COLS)
print(f'X: {X.shape} | y mean: {y.mean():.2f}')


Loaded parquet files from /Users/bashkal/Desktop/ML/ML-Final/Internship/outputs
X: (29008, 206) | y mean: 16.22


In [14]:
# Identify numeric vs categorical columns
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

# Split categoricals into low-card (OHE) and high-card (Target Encoding)
# Threshold = 15 levels for OHE, else Target Encoder
cat_low  = [c for c in cat_cols if X[c].nunique() <= 15]
cat_high = [c for c in cat_cols if X[c].nunique() > 15]

print(f'Numeric cols    : {len(num_cols)}')
print(f'Low-card cats   : {len(cat_low)}')
print(f'High-card cats  : {len(cat_high)}')

Numeric cols    : 205
Low-card cats   : 0
High-card cats  : 1


### KFoldTargetEncoder (Module-7 style custom transformer, repeated here for self-containment)

In [15]:
class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, n_splits=5, smoothing=20.0, random_state=42):
        self.cols = cols; self.n_splits = n_splits
        self.smoothing = smoothing; self.random_state = random_state

    def _smoothed_map(self, X_col, y):
        stats = pd.DataFrame({'cat': X_col, 'y': y}).groupby('cat')['y'].agg(['mean', 'count'])
        return ((stats['count'] * stats['mean'] + self.smoothing * self.global_mean_)
                / (stats['count'] + self.smoothing)).to_dict()

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.global_mean_ = float(y.mean())
        self.maps_ = {c: self._smoothed_map(X[c].astype(str).fillna('__nan__'), y)
                      for c in self.cols}
        return self

    def transform(self, X):
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = (X[c].astype(str).fillna('__nan__').map(self.maps_[c])
                       .fillna(self.global_mean_).astype('float32'))
        return Xo

    def fit_transform(self, X, y=None, **kw):
        y = np.asarray(y, dtype=float); self.global_mean_ = float(y.mean())
        Xo = X.copy()
        for c in self.cols: Xo[c] = np.full(len(X), self.global_mean_, dtype='float32')
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        for tr, va in kf.split(X):
            for c in self.cols:
                m = self._smoothed_map(X[c].astype(str).fillna('__nan__').iloc[tr], y[tr])
                mapped = X[c].astype(str).fillna('__nan__').iloc[va].map(m)
                Xo.iloc[va, Xo.columns.get_loc(c)] = mapped.fillna(self.global_mean_).astype('float32').values
        self.maps_ = {c: self._smoothed_map(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return Xo

    def get_feature_names_out(self, input_features=None):
        return np.array(input_features if input_features is not None else self.cols)
print('KFoldTargetEncoder ready.')

KFoldTargetEncoder ready.


## 3. Preprocessing `ColumnTransformer`
MLPs require **scaled** numeric inputs — that's the key difference vs. tree-based pipelines.

In [16]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler',  StandardScaler()),
])
cat_low_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
cat_high_pipe = Pipeline([
    ('te', KFoldTargetEncoder(cols=cat_high, n_splits=5, smoothing=20, random_state=RANDOM_STATE)),
    ('scaler', StandardScaler()),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols),
    ('cat_low', cat_low_pipe, cat_low),
    ('cat_high', cat_high_pipe, cat_high),
])
print('Preprocessor ready.')

Preprocessor ready.


## 4. MLP Architecture
Module 12 covers MLPs with Linear → activation → (BatchNorm/Dropout). Module 13 covers Adam.

We use:
- 3 hidden layers (256 → 128 → 64) — a moderate-sized network for ~120-200 features.
- ReLU activations + BatchNorm for stable training + Dropout for regularisation.
- Output layer: linear (regression).
- Loss: MSE.

In [17]:
class MLPRegressor(nn.Module):
    def __init__(self, in_dim, hidden=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

## 5. One-Fold Training Utility
Standard training loop with early stopping on validation MSE.

In [18]:
def train_one_fold(X_tr, y_tr, X_va, y_va, *,
                   max_epochs=60, batch_size=512, lr=1e-3, weight_decay=1e-5,
                   patience=8, verbose=False):
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
    y_tr_t = torch.tensor(np.log1p(y_tr), dtype=torch.float32)   # log1p target
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
    y_va_arr = np.asarray(y_va, dtype=float)

    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    model = MLPRegressor(in_dim=X_tr.shape[1]).to(device)
    optim = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_mse = float('inf'); best_state = None; bad = 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in dl:
            xb = xb.to(device); yb = yb.to(device)
            optim.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optim.step()

        model.eval()
        with torch.no_grad():
            pred_va = model(X_va_t).cpu().numpy()
        pred_va = np.clip(np.expm1(pred_va), 0, 90)            # invert log1p
        va_mse = mean_squared_error(y_va_arr, pred_va)

        if va_mse < best_mse - 1e-4:
            best_mse = va_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
        if verbose:
            print(f'    epoch {epoch:3d} | val MSE = {va_mse:.3f} | best = {best_mse:.3f}')
        if bad >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        final_va = np.clip(np.expm1(model(X_va_t).cpu().numpy()), 0, 90)
    return model, best_mse, final_va

## 6. 5-Fold CV (consistent split with notebooks 03 / 04)

In [19]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(len(y))
fold_mse = []
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    print(f'\n--- Fold {fold+1}/{N_SPLITS} ---')
    # Fit preprocessor on training fold, transform both
    pp = ColumnTransformer(transformers=[
        ('num', num_pipe, num_cols),
        ('cat_low', cat_low_pipe, cat_low),
        ('cat_high', cat_high_pipe, cat_high),
    ])
    X_tr_proc = pp.fit_transform(X.iloc[tr_idx], y[tr_idx]).astype('float32')
    X_va_proc = pp.transform(X.iloc[va_idx]).astype('float32')
    print(f'  features after preprocessing: {X_tr_proc.shape[1]}')

    model, va_mse, va_pred = train_one_fold(
        X_tr_proc, y[tr_idx], X_va_proc, y[va_idx], verbose=False,
    )
    oof[va_idx] = va_pred
    fold_mse.append(va_mse)
    print(f'  fold MSE = {va_mse:.3f}')

print(f'\nMLP | CV MSE = {np.mean(fold_mse):.3f} ± {np.std(fold_mse):.3f} | {(time.time()-t0)/60:.1f} min')
np.save(OUT_DIR / 'oof_MLP.npy', oof)


--- Fold 1/5 ---
  features after preprocessing: 273
  fold MSE = 383.804

--- Fold 2/5 ---
  features after preprocessing: 273
  fold MSE = 386.449

--- Fold 3/5 ---
  features after preprocessing: 273
  fold MSE = 368.909

--- Fold 4/5 ---
  features after preprocessing: 273
  fold MSE = 382.571

--- Fold 5/5 ---
  features after preprocessing: 269
  fold MSE = 390.745

MLP | CV MSE = 382.496 ± 7.348 | 2.0 min


## 7. Final Evaluation on Held-Out Test Set
Refit one MLP per fold on full training data, average fold predictions on the local 20% test set, and record the held-out MSE.

In [20]:
pp_full = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols),
    ('cat_low', cat_low_pipe, cat_low),
    ('cat_high', cat_high_pipe, cat_high),
])
X_full_proc = pp_full.fit_transform(X, y).astype('float32')
print('features after preprocessing:', X_full_proc.shape[1])

# Use the last 10% as a tiny holdout for early stopping during the full-train fits
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.permutation(len(y))
split = int(0.9 * len(y))
tr_idx_full, va_idx_full = idx[:split], idx[split:]

print('Full-train preprocessing done.')

features after preprocessing: 273
Full-train preprocessing done.


## Summary
- 3-layer PyTorch MLP (256→128→64) trained with **Adam** on `log1p(y)`.
- StandardScaler preprocessing; all features are numeric so no categorical encoding needed.
- Early stopping on validation MSE prevents overfit.
- OOF predictions stored as `oof_MLP.npy` for the ensemble blender.
- Local test predictions stored as `test_local_pred_MLP.npy`.

### → Next: `07_Blend.ipynb`
Combine OOFs from all five models with SLSQP-optimized convex weights.

In [21]:
from sklearn.metrics import mean_absolute_error, r2_score

mlp_mse = mean_squared_error(y, oof)
mlp_mae = mean_absolute_error(y, oof)
mlp_r2  = r2_score(y, oof)

print(f"MLP Performance (CV):")
print(f"MSE: {mlp_mse:.3f}")
print(f"MAE: {mlp_mae:.3f}")
print(f"R2 : {mlp_r2:.3f}")

# Update results table if exists
try:
    results_df = pd.read_csv(OUT_DIR / 'baseline_cv_results.csv')
    # Use standard concatenation/update logic
    new_row = pd.DataFrame([{
        'name': 'MLP',
        'mse_mean': mlp_mse,
        'mse_std': 0.0, # Placeholder
        'mae_mean': mlp_mae,
        'r2_mean': mlp_r2,
        'seconds': 0.0 # Placeholder
    }])
    results_df = pd.concat([results_df, new_row], ignore_index=True).drop_duplicates('name', keep='last')
    results_df.to_csv(OUT_DIR / 'baseline_cv_results.csv', index=False)
    print("Updated baseline_cv_results.csv with MLP.")
except Exception as e:
    print(f"Could not update results CSV: {e}")

MLP Performance (CV):
MSE: 382.495
MAE: 10.401
R2 : 0.393
Updated baseline_cv_results.csv with MLP.


## 8. Section IV.C — Unbiased Local Test Evaluation
Evaluating the MLP ensemble on the 20% local test set.


In [22]:
# 7. Evaluate on Local Test Set (Section IV.C)
from sklearn.metrics import mean_absolute_error, r2_score

# Refit the MLP on the full training set using the exact same preprocessing pipeline
# that will be applied to the local test set.
model_full, _, _ = train_one_fold(
    X_full_proc[tr_idx_full], y[tr_idx_full],
    X_full_proc[va_idx_full], y[va_idx_full],
    verbose=False,
)
model_full.eval()

X_test_local = test_local_df.drop(columns=DROP_COLS)
X_test_local = X_test_local.reindex(columns=X.columns, fill_value=0).reset_index(drop=True)
y_test_local = test_local_df[TARGET].astype(float).values

# Transform the local test set with the same fitted preprocessor used for training.
X_test_local_proc = pp_full.transform(X_test_local).astype(np.float32)
assert X_test_local_proc.shape[1] == X_full_proc.shape[1], (
    f"Feature mismatch: train has {X_full_proc.shape[1]} features, test has {X_test_local_proc.shape[1]}"
)
assert X_test_local_proc.shape[1] == model_full.net[0].in_features

test_local_tensor = torch.FloatTensor(X_test_local_proc).to(device)

with torch.no_grad():
    # Model predicts in log-space, so we must invert with expm1
    test_local_pred = np.expm1(model_full(test_local_tensor).cpu().numpy().flatten())
    test_local_pred = np.clip(test_local_pred, 0, 90)

model = model_full

test_local_mse = mean_squared_error(y_test_local, test_local_pred)
test_local_mae = mean_absolute_error(y_test_local, test_local_pred)
test_local_r2  = r2_score(y_test_local, test_local_pred)

print(f"--- Unbiased Local Test Results (MLP) ---")
print(f"Local Test MSE: {test_local_mse:.3f}")
print(f"Local Test MAE: {test_local_mae:.3f}")
print(f"Local Test R2 : {test_local_r2:.3f}")

# Save for blend
np.save(OUT_DIR / 'test_local_pred_mlp.npy', test_local_pred)

--- Unbiased Local Test Results (MLP) ---
Local Test MSE: 377.314
Local Test MAE: 10.232
Local Test R2 : 0.392
